In [ ]:
#Importamos pandas y sqlite3
import pandas as pd
import sqlite3

#Cargamos nuestro archivo limpio que guardamos en la Fase 1
df_limpio = pd.read_csv('../data/telco_churn_limpio.csv')

#Creamos una conexión a una base de datos de SQLite local
conn = sqlite3.connect('../data/telco_db.sqlite')

#Cargamos el DataFrame como una tabla SQL llamada 'clientes'
df_limpio.to_sql('clientes', conn, if_exists='replace', index=False)

print("¡Base de datos SQL creada con éxito! La tabla 'clientes' está lista para consultas.")

¡Base de datos SQL creada con éxito! La tabla 'clientes' está lista para consultas.


In [ ]:
# Consulta para calcular el total de clientes y la cantidad de fugas
query = """
    SELECT 
        COUNT(*) AS Total_Clientes,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS Clientes_Fugados,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Porcentaje_Fuga
    FROM clientes
"""

# Ejecutamos la consulta
df_resultado = pd.read_sql(query, conn)
df_resultado

,Total_Clientes,Clientes_Fugados,Porcentaje_Fuga
0,7043,1869,26.54


In [ ]:
#Consulta para analizar la fuga según el tipo de contrato
query_contrato = """
    SELECT 
        Contract,
        COUNT(*) AS Total_Clientes,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS Fugados,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Porcentaje_Fuga
    FROM clientes
    GROUP BY Contract
    ORDER BY Porcentaje_Fuga DESC
"""

df_contrato = pd.read_sql(query_contrato, conn)
df_contrato

,Contract,Total_Clientes,Fugados,Porcentaje_Fuga
0,Month-to-month,3875,1655,42.71
1,One year,1473,166,11.27
2,Two year,1695,48,2.83


In [ ]:
#Consulta para analizar la fuga según el tipo de servicio de internet
query_internet = """
    SELECT 
        InternetService,
        COUNT(*) AS Total_Clientes,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS Fugados,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Porcentaje_Fuga,
        ROUND(AVG(MonthlyCharges), 2) AS Promedio_Cargo_Mensual
    FROM clientes
    GROUP BY InternetService
    ORDER BY Porcentaje_Fuga DESC
"""

df_internet = pd.read_sql(query_internet, conn)
df_internet

,InternetService,Total_Clientes,Fugados,Porcentaje_Fuga,Promedio_Cargo_Mensual
0,Fiber optic,3096,1297,41.89,91.50
1,DSL,2421,459,18.96,58.10
2,No,1526,113,7.40,21.08


In [ ]:
#Consulta avanzada cruzando Método de Pago y Fuga
query_pago = """
    SELECT 
        PaymentMethod,
        COUNT(*) AS Total_Clientes,
        SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) AS Fugados,
        ROUND(SUM(CASE WHEN Churn = 'Yes' THEN 1 ELSE 0 END) * 100.0 / COUNT(*), 2) AS Porcentaje_Fuga,
        ROUND(AVG(tenure), 1) AS Promedio_Meses_Antiguedad
    FROM clientes
    GROUP BY PaymentMethod
    ORDER BY Porcentaje_Fuga DESC
"""

df_pago = pd.read_sql(query_pago, conn)
df_pago

,PaymentMethod,Total_Clientes,Fugados,Porcentaje_Fuga,Promedio_Meses_Antiguedad
0,Electronic check,2365,1071,45.29,25.2
1,Mailed check,1612,308,19.11,21.8
2,Bank transfer (automatic),1544,258,16.71,43.7
3,Credit card (automatic),1522,232,15.24,43.3


In [ ]:
#Consulta
query_avanzada = """
    SELECT 
        customerID,
        Contract,
        MonthlyCharges,
        -- Calculamos el promedio del cargo mensual según el tipo de contrato para cada fila
        ROUND(AVG(MonthlyCharges) OVER(PARTITION BY Contract), 2) AS Promedio_Contrato,
        -- Evaluamos si el cliente paga más que el promedio de su mismo tipo de contrato
        CASE 
            WHEN MonthlyCharges > AVG(MonthlyCharges) OVER(PARTITION BY Contract) THEN 'Superior al Promedio'
            ELSE 'Normal o Inferior'
        END AS Categoria_Precio,
        Churn
    FROM clientes
    LIMIT 10
"""

df_avanzado = pd.read_sql(query_avanzada, conn)
df_avanzado

,customerID,Contract,MonthlyCharges,Promedio_Contrato,Categoria_Precio,Churn
0,7590-VHVEG,Month-to-month,29.85,66.4,Normal o Inferior,No
1,3668-QPYBK,Month-to-month,53.85,66.4,Normal o Inferior,Yes
2,9237-HQITU,Month-to-month,70.70,66.4,Superior al Promedio,Yes
3,9305-CDSKC,Month-to-month,99.65,66.4,Superior al Promedio,Yes
4,1452-KIOVK,Month-to-month,89.10,66.4,Superior al Promedio,No
5,6713-OKOMC,Month-to-month,29.75,66.4,Normal o Inferior,No
6,7892-POOKP,Month-to-month,104.80,66.4,Superior al Promedio,Yes
7,9763-GRSKD,Month-to-month,49.95,66.4,Normal o Inferior,No
8,0280-XJGEX,Month-to-month,103.70,66.4,Superior al Promedio,Yes
9,5129-JLPIS,Month-to-month,105.50,66.4,Superior al Promedio,No


In [7]:
# Cerramos la conexión a SQLite
conn.close()
print("¡Conexión cerrada y Fase 2 de SQL finalizada con éxito!")

¡Conexión cerrada y Fase 2 de SQL finalizada con éxito!
